<a href="https://colab.research.google.com/github/roshanth155/docuqa-rag/blob/main/docuqa_rag.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Step 1: Install (run this first)

In [1]:
!pip install -q rank_bm25 sentence-transformers pypdf gradio transformers


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 407.8/407.8 kB 12.6 MB/s eta 0:00:00


## Step 2: Load models and define functions (upload syllabus.pdf and qa.csv BEFORE running)

In [2]:
import re, time
import numpy as np
import pandas as pd
from pypdf import PdfReader
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer, CrossEncoder, util

PDF, QA = "syllabus.pdf", "qa.csv"

embedder = SentenceTransformer("BAAI/bge-small-en-v1.5")
reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

def tok(t): return re.findall(r"\w+", t.lower())
def norm(t): return re.sub(r"\s+", " ", t.lower()).strip()

def load_pages(pdf):
    return [(i + 1, p.extract_text() or "") for i, p in enumerate(PdfReader(pdf).pages)]

def chunk(pages, size, overlap=0.2):
    out, step = [], max(1, int(size * (1 - overlap)))
    for pg, text in pages:
        w = text.split()
        for s in range(0, max(len(w), 1), step):
            c = " ".join(w[s:s + size])
            if len(c.split()) > 15:
                out.append({"page": pg, "text": c})
    return out

class Index:
    def __init__(self, chunks):
        self.c = chunks
        self.texts = [c["text"] for c in chunks]
        self.bm = BM25Okapi([tok(t) for t in self.texts])
        self.emb = embedder.encode(self.texts, normalize_embeddings=True, convert_to_tensor=True)

    def bm25(self, q): return np.array(self.bm.get_scores(tok(q)))

    def dense(self, q):
        qe = embedder.encode("Represent this sentence for searching relevant passages: " + q,
                             normalize_embeddings=True, convert_to_tensor=True)
        return util.cos_sim(qe, self.emb)[0].cpu().numpy()

    def hybrid(self, q, a=0.5):
        z = lambda x: (x - x.mean()) / (x.std() + 1e-9)
        return a * z(self.dense(q)) + (1 - a) * z(self.bm25(q))

    def search(self, q, method, k):
        if method == "rerank":
            cand = np.argsort(-self.hybrid(q))[:20]
            sc = reranker.predict([(q, self.texts[i]) for i in cand])
            order = np.argsort(-sc)[:k]
            return [int(cand[i]) for i in order], [float(sc[i]) for i in order]
        sc = getattr(self, method)(q)
        idx = np.argsort(-sc)[:k]
        return [int(i) for i in idx], [float(sc[i]) for i in idx]


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/743 [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/1.33k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

## Step 3: Evaluation results

In [3]:
pages = load_pages(PDF)
qa = pd.read_csv(QA)
qa_ans = qa[qa["answerable"] == 1].reset_index(drop=True)

def evaluate(index, method, ks=(1, 3, 5, 10)):
    ranks, t0 = [], time.time()
    for _, r in qa_ans.iterrows():
        idx, _ = index.search(r["question"], method, max(ks))
        rank = next((i + 1 for i, j in enumerate(idx)
                     if norm(str(r["snippet"])) in norm(index.texts[j])), None)
        ranks.append(rank)
    lat = (time.time() - t0) / len(qa_ans) * 1000
    res = {f"Recall@{k}": np.mean([r is not None and r <= k for r in ranks]) for k in ks}
    res["MRR"] = np.mean([1 / r if r else 0 for r in ranks])
    res["Latency(ms/query)"] = lat
    return res

methods = ["bm25", "dense", "hybrid", "rerank"]
rows = []
for size in [128, 256, 512]:                      # chunk-size ablation
    idx = Index(chunk(pages, size))
    for m in methods:
        rows.append({"chunk_size": size, "method": m, **evaluate(idx, m)})
results = pd.DataFrame(rows).round(3)
print(results.to_string(index=False))
results.to_csv("results.csv", index=False)         # paste/screenshot into report


 chunk_size method  Recall@1  Recall@3  Recall@5  Recall@10   MRR  Latency(ms/query)
        128   bm25     0.525     0.675     0.750      0.825 0.624              1.052
        128  dense     0.525     0.725     0.825      0.900 0.649             12.088
        128 hybrid     0.575     0.775     0.925      0.925 0.695             11.883
        128 rerank     0.725     0.875     0.900      0.950 0.807             60.185
        256   bm25     0.600     0.775     0.825      0.900 0.703              0.850
        256  dense     0.500     0.825     0.875      0.975 0.667             12.417
        256 hybrid     0.575     0.875     0.925      0.975 0.731             12.448
        256 rerank     0.600     0.825     0.900      0.950 0.726            104.898
        512   bm25     0.625     0.875     0.900      0.950 0.746              0.886
        512  dense     0.450     0.725     0.800      0.950 0.611             11.873
        512 hybrid     0.625     0.875     0.925      0.950 0.756

## Step 4: Refusal check

In [6]:
BEST = Index(chunk(pages, 128))                    # change to your best chunk size
THRESH = 0.65                                      # tune this: dense cosine cutoff

def top_dense_score(q):
    return float(np.max(BEST.dense(q)))

ans_scores = [top_dense_score(q) for q in qa[qa.answerable == 1].question]
oos_scores = [top_dense_score(q) for q in qa[qa.answerable == 0].question]
print("Avg top score, answerable:", np.mean(ans_scores).round(3))
print("Avg top score, out-of-scope:", np.mean(oos_scores).round(3))
refused_oos = np.mean([s < THRESH for s in oos_scores])
wrongly_refused = np.mean([s < THRESH for s in ans_scores])
print(f"Correctly refused out-of-scope: {refused_oos:.2%}")
print(f"Wrongly refused answerable:     {wrongly_refused:.2%}")
# Try THRESH values 0.45 / 0.55 / 0.65 and report the trade-off table.


Avg top score, answerable: 0.757
Avg top score, out-of-scope: 0.586
Correctly refused out-of-scope: 88.89%
Wrongly refused answerable:     2.50%


In [5]:
rows = []
for t in [0.55, 0.60, 0.65, 0.68, 0.70, 0.72, 0.75]:
    rows.append({
        "THRESH": t,
        "Correctly refused OOS": round(np.mean([s < t for s in oos_scores]), 3),
        "Wrongly refused answerable": round(np.mean([s < t for s in ans_scores]), 3),
    })
thresh_table = pd.DataFrame(rows)
print(thresh_table.to_string(index=False))
thresh_table.to_csv("threshold_table.csv", index=False)

 THRESH  Correctly refused OOS  Wrongly refused answerable
   0.55                  0.111                       0.000
   0.60                  0.778                       0.000
   0.65                  0.889                       0.025
   0.68                  0.889                       0.050
   0.70                  1.000                       0.125
   0.72                  1.000                       0.275
   0.75                  1.000                       0.500


## Step 5: Demo

In [9]:
try: demo.close()
except Exception: pass

import torch, gradio as gr
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

dev = "cuda" if torch.cuda.is_available() else "cpu"
t5_tok = AutoTokenizer.from_pretrained("google/flan-t5-large")
t5 = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-large").to(dev)

def gen(prompt):
    inputs = t5_tok(prompt, return_tensors="pt", truncation=True, max_length=1024).to(dev)
    out = t5.generate(**inputs, max_new_tokens=100, num_beams=3)
    return t5_tok.decode(out[0], skip_special_tokens=True)

def answer(question):
    if top_dense_score(question) < THRESH:
        return "Not found in the document.", ""
    idx, _ = BEST.search(question, "rerank", 3)
    ctx = "\n\n".join(BEST.texts[i] for i in idx)
    prompt = (f"Read the rules below and answer the question in one complete sentence. "
              f"Use only the rules.\n\nRules:\n{ctx}\n\nQuestion: {question}\nAnswer:")
    out = gen(prompt)
    src = "\n\n".join(f"[Page {BEST.c[i]['page']}] {BEST.texts[i]}" for i in idx)
    return out, src

def rebuild(file):
    global BEST
    BEST = Index(chunk(load_pages(file.name), 256))
    return "Document indexed."

with gr.Blocks() as demo:
    gr.Markdown("## Document QA Assistant (RAG)")
    f = gr.File(label="Upload PDF"); status = gr.Textbox(label="Status")
    f.upload(rebuild, f, status)
    q = gr.Textbox(label="Ask a question")
    a = gr.Textbox(label="Answer"); s = gr.Textbox(label="Cited sources", lines=12)
    gr.Button("Ask").click(answer, q, [a, s])
demo.launch(share=True)

Closing server running on port: 7860


config.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.20k [00:00<?, ?B/s]

...

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/558 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://3f40624417c0c730e9.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [10]:
import html, time
import gradio as gr

try: demo.close()
except Exception: pass

CSS = """
.gradio-container {max-width: 980px !important; margin: auto;
  background: radial-gradient(1200px 500px at 10% -10%, #3b2a8f33, transparent),
              radial-gradient(900px 400px at 100% 0%, #06b6d433, transparent);}
.hero {padding: 28px 32px; border-radius: 20px; color: #fff;
  background: linear-gradient(135deg, #4f46e5 0%, #7c3aed 50%, #06b6d4 100%);
  box-shadow: 0 10px 40px #4f46e555;}
.hero h1 {margin: 0 0 6px 0; font-size: 30px; font-weight: 800; letter-spacing: -0.5px;}
.hero p {margin: 0; opacity: .92; font-size: 15px;}
.pills {margin-top: 16px; display: flex; flex-wrap: wrap; gap: 8px;}
.pill {background: #ffffff26; border: 1px solid #ffffff55; padding: 5px 12px;
  border-radius: 999px; font-size: 12.5px; font-weight: 600; backdrop-filter: blur(6px);}
.card {border-radius: 16px; padding: 18px 20px; margin-top: 6px;
  border: 1px solid #8b5cf655; background: #7c3aed12;}
.card .label {font-size: 11.5px; font-weight: 700; letter-spacing: 1.2px;
  text-transform: uppercase; color: #8b5cf6; margin-bottom: 8px;}
.card .ans {font-size: 18px; line-height: 1.5; font-weight: 600;}
.card.refuse {border-color: #f59e0b88; background: #f59e0b14;}
.card.refuse .label {color: #f59e0b;}
.meta {display: flex; gap: 8px; margin-top: 12px; flex-wrap: wrap;}
.chip {font-size: 12px; padding: 4px 10px; border-radius: 8px;
  background: #06b6d422; border: 1px solid #06b6d466; font-weight: 600;}
.src {border-radius: 14px; padding: 14px 16px; margin-top: 10px;
  border: 1px solid #94a3b855; background: #94a3b814;}
.src .top {display: flex; justify-content: space-between; align-items: center; margin-bottom: 8px;}
.badge {background: linear-gradient(135deg,#4f46e5,#7c3aed); color: #fff;
  font-size: 12px; font-weight: 700; padding: 3px 10px; border-radius: 999px;}
.sc {font-size: 12px; opacity: .75; font-weight: 600;}
.src .txt {font-size: 13.5px; line-height: 1.55; opacity: .92; max-height: 150px; overflow-y: auto;}
.idle {opacity: .6; padding: 20px; text-align: center; border: 1px dashed #94a3b877; border-radius: 14px;}
#askbtn {background: linear-gradient(135deg,#4f46e5,#7c3aed) !important; color: #fff !important;
  font-weight: 700 !important; border: none !important; border-radius: 12px !important;}
"""

HERO = """
<div class="hero">
  <h1>📚 DocuQA &mdash; Ask Your Rulebook</h1>
  <p>Document-grounded question answering with cited sources and a hallucination guard.</p>
  <div class="pills">
    <span class="pill">🔎 Hybrid BM25 + Dense retrieval</span>
    <span class="pill">🎯 Cross-encoder reranker</span>
    <span class="pill">🧠 Flan-T5-large generator</span>
    <span class="pill">🛡️ Refusal guard</span>
  </div>
</div>
"""

IDLE = '<div class="idle">Upload a PDF, then ask a question. The answer and its sources will appear here.</div>'
esc = html.escape

def answer(question):
    if not question or not question.strip():
        return IDLE, ""
    t0 = time.time()
    conf = top_dense_score(question)
    if conf < THRESH:
        card = (f'<div class="card refuse"><div class="label">Not found</div>'
                f'<div class="ans">This question doesn\'t seem to be covered in the document.</div>'
                f'<div class="meta"><span class="chip">similarity {conf:.2f} &lt; threshold {THRESH}</span>'
                f'<span class="chip">⏱ {(time.time()-t0)*1000:.0f} ms</span></div></div>')
        return card, ""
    idx, sc = BEST.search(question, "rerank", 3)
    ctx = "\n\n".join(BEST.texts[i] for i in idx)
    prompt = (f"Read the rules below and answer the question in one complete sentence. "
              f"Use only the rules.\n\nRules:\n{ctx}\n\nQuestion: {question}\nAnswer:")
    out = gen(prompt)
    ms = (time.time() - t0) * 1000
    card = (f'<div class="card"><div class="label">Answer</div><div class="ans">{esc(out)}</div>'
            f'<div class="meta"><span class="chip">confidence {conf:.2f}</span>'
            f'<span class="chip">📄 {len(idx)} sources</span>'
            f'<span class="chip">⏱ {ms/1000:.1f} s</span></div></div>')
    srcs = ""
    for rank, (i, s) in enumerate(zip(idx, sc), 1):
        srcs += (f'<div class="src"><div class="top"><span class="badge">Source {rank} &middot; Page {BEST.c[i]["page"]}</span>'
                 f'<span class="sc">rerank score {s:.2f}</span></div>'
                 f'<div class="txt">{esc(BEST.texts[i])}</div></div>')
    return card, srcs

def rebuild(file):
    global BEST
    BEST = Index(chunk(load_pages(file.name), 256))
    return "✅ Document indexed. Ask away!"

EXAMPLES = ["What is the minimum attendance required?",
            "How many activity points must regular UG students earn?",
            "When is the I grade awarded?",
            "What is the plagiarism limit for project reports?",
            "What are the canteen timings?"]

def build(**kw):
    with gr.Blocks(**kw) as d:
        gr.HTML(HERO)
        with gr.Accordion("📎 Upload document (PDF)", open=True):
            f = gr.File(label="Drop your PDF here", file_types=[".pdf"])
            status = gr.Textbox(label="Status", interactive=False, value="Waiting for a PDF...")
        f.upload(rebuild, f, status)
        q = gr.Textbox(label="Your question", placeholder="e.g. What happens if my attendance is below 85%?", lines=1)
        btn = gr.Button("Ask ✨", elem_id="askbtn")
        gr.Examples(EXAMPLES, inputs=q, label="Try these")
        a = gr.HTML(IDLE)
        gr.Markdown("### 📄 Cited sources")
        s = gr.HTML("")
        btn.click(answer, q, [a, s]); q.submit(answer, q, [a, s])
    return d

theme = gr.themes.Soft(primary_hue="violet", secondary_hue="cyan")
try:
    demo = build(theme=theme, css=CSS)
    demo.launch(share=True)
except TypeError:
    demo = build()
    demo.launch(share=True, theme=theme, css=CSS)

Closing server running on port: 7860


/tmp/ipykernel_875/792254067.py:97: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme, css. Please pass these parameters to launch() instead.
  with gr.Blocks(**kw) as d:


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://fafa0cd15fecb44f10.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
